In [2]:
import os
import numpy as np
import pandas as pd

ROOT = os.environ.get("AGL_ROOT", ".")


def p(*parts):
    return os.path.join(ROOT, *parts)


LEVELS_FILE = p("Code Outputs", "Gap Interpolation Outputs",
                "Unified_Interpolated_Levels.xlsx")
METRIC_FILES = [
    p("Code Outputs", "Arima Forecast Outputs", "FC_metrics.csv"),
    p("Code Outputs", "Forecast Var Outputs", "FC4_metrics.csv"),
    p("Code Outputs", "SARIMAX Climate Outputs", "FC5_climate_metrics.csv"),
    p("Code Outputs", "GNN Outputs", "FC6_gnn_metrics.csv"),
    p("Code Outputs", "VARX Diagnostic Outputs", "VARX_diagnostic_metrics.csv"),
    p("Code Outputs", "Baseline Outputs", "Baseline_metrics.csv"),
    p("Code Outputs", "GNARX Outputs", "FC7_gnarx_metrics.csv"),
    p("Code Outputs", "GNARX Outputs", "FC8_adjacency_sensitivity.csv"),
    p("Code Outputs", "GNN Outputs", "FC9_gnn_adjacency.csv"),
]


START, END = "1995-06-01", "2025-12-01"
TEST_MONTHS = 60
SEASONAL_PERIOD = 12      # 12 = seasonal-naive scale; 1 = random-walk scale


# Per-lake denominators from the training portion of the canonical window

lev = pd.read_excel(LEVELS_FILE)
lev["Date"] = pd.to_datetime(lev["Date"])
L = (lev.pivot(index="Date", columns="Reservoir", values="Level_m")
       .sort_index().asfreq("MS").loc[START:END])
assert L.notna().all().all(), "NaNs inside the canonical window"

split = len(L) - TEST_MONTHS
train = L.iloc[:split]                      # contiguous monthly index preserved
m = SEASONAL_PERIOD

den = {}
for lk in L.columns:
    y = train[lk].to_numpy(dtype=float)
    den[lk] = {
        "sd": float(np.nanstd(y, ddof=1)),
        "mase_scale": float(np.nanmean(np.abs(y[m:] - y[:-m]))),
    }

print("=== PER-LAKE DENOMINATORS ===")
print(f"  training portion: {train.index[0].date()} .. {train.index[-1].date()} "
      f"({split} months)")
print(f"{'Lake':17s} {'std_train_m':>12s} {'MASE_scale_m':>13s}  (m={m})")
for lk, d in den.items():
    print(f"{lk:17s} {d['sd']:12.3f} {d['mase_scale']:13.3f}")


# Augment each metric file

def augment(path):
    if not os.path.exists(path):
        print(f"  [skip] {path} not found")
        return None
    df = pd.read_csv(path)
    if "Lake" not in df.columns or "RMSE_m" not in df.columns:
        print(f"  [skip] {path}: no Lake/RMSE_m columns")
        return None

    df["nRMSE_pct"] = df.apply(
        lambda r: round(100 * r["RMSE_m"] / den[r["Lake"]]["sd"], 1)
        if r["Lake"] in den else np.nan, axis=1)

    if "MAE_m" in df.columns:
        df["MASE"] = df.apply(
            lambda r: round(r["MAE_m"] / den[r["Lake"]]["mase_scale"], 3)
            if r["Lake"] in den else np.nan, axis=1)
    else:
        # some tables (e.g. the VARX diagnostic) carry RMSE only
        df["MASE"] = np.nan
        print(f"  [note] {os.path.basename(path)} has no MAE_m column -> MASE left blank")

    out_path = path.replace(".csv", "_scaled.csv")
    df.to_csv(out_path, index=False)
    print(f"  wrote {out_path}  (+nRMSE_pct, +MASE)")
    return df


print("\n=== AUGMENTING METRIC FILES ===")
augmented = {path: augment(path) for path in METRIC_FILES}

for path, df in augmented.items():
    if df is None:
        continue
    name = os.path.basename(path)
    
    if "Graph" in df.columns and "Model" in df.columns:
        idx_col = ["Graph", "Model"]
    elif "Graph" in df.columns:
        idx_col = "Graph"
    else:
        idx_col = "Model"

    if df["MASE"].notna().any():
        print(f"\n=== {name}: mean MASE by {idx_col} x Horizon (<1 beats seasonal naive) ===")
        print(df.pivot_table(index=idx_col, columns="Horizon_m",
                             values="MASE").round(2).to_string())
    print(f"--- {name}: mean nRMSE_pct by {idx_col} x Horizon (lower is better) ---")
    print(df.pivot_table(index=idx_col, columns="Horizon_m",
                         values="nRMSE_pct").round(0).to_string())

print("\nDone.")

=== PER-LAKE DENOMINATORS ===
  training portion: 1995-06-01 .. 2020-12-01 (307 months)
Lake               std_train_m  MASE_scale_m  (m=12)
Lake Albert              0.567         0.425
Lake Edward              0.307         0.299
Lake Kivu                0.327         0.292
Lake Malawi              0.644         0.290
Lake Tanganyika          0.469         0.296
Lake Turkana             1.128         0.667
Lake Victoria            0.551         0.317

=== AUGMENTING METRIC FILES ===
  wrote ./Code Outputs/Arima Forecast Outputs/FC_metrics_scaled.csv  (+nRMSE_pct, +MASE)
  wrote ./Code Outputs/Forecast Var Outputs/FC4_metrics_scaled.csv  (+nRMSE_pct, +MASE)
  wrote ./Code Outputs/SARIMAX Climate Outputs/FC5_climate_metrics_scaled.csv  (+nRMSE_pct, +MASE)
  wrote ./Code Outputs/GNN Outputs/FC6_gnn_metrics_scaled.csv  (+nRMSE_pct, +MASE)
  wrote ./Code Outputs/VARX Diagnostic Outputs/VARX_diagnostic_metrics_scaled.csv  (+nRMSE_pct, +MASE)
  wrote ./Code Outputs/Baseline Outputs/Baseline_